# 03 — Geometric Localisation (Hough Circle Transform)

This notebook implements the Hough Circle Transform (HCT) approach to detecting and localising circular stamps. It builds on the HSV + morphology preprocessing from notebook 02 and attempts to fit circles to the cleaned edge map.

> **Note:** This is an exploratory notebook documenting the Hough Circle Transform approach that was tested before settling on the contour-based method. HCT was sensitive to imperfect circles, partially occluded stamps, and uneven ink edges — causing it to fail on several images in the dataset. The contour-based approach in `03_stamp_segmentation_contours.ipynb` produced more consistent results across varying stamp conditions and was selected for the final pipeline. Both notebooks are kept to document the experimentation process.

**Input:** Raw scanned document images  
**Output:** Circle candidates with centre (x, y) and radius r, scored by ring density

---
## Pipeline Overview
```
Image → HSV mask → Morphological cleaning → Canny edges → 
HoughCircles → Ring score filtering → Ranked candidates
```

---
## Contents
1. Imports
2. Helper: `show_image`
3. Helper: `ring_score` — validate circle quality
4. Function: `detect_colored_components` — HSV + morphology + connected components
5. Function: `visualize_components`
6. Test on single image — component detection
7. Function: `crop_component_region` — extract region around a component
8. Visualise component crop
9. HSV mask + Canny edges on crop
10. Apply Hough Circle Transform on crop
11. Function: `crop_top_components`
12. Visualise top candidate crops
13. Function: `detect_stamp_candidates` — full pipeline
14. Function: `visualize_candidates`
15. Test on single image — full pipeline
16. Batch test on 5 genuine + 5 forged images


## 1. Imports

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 2. Helper: `show_image`

Convenience wrapper around `matplotlib.pyplot` for displaying single images with a title.

In [ ]:
def show_image(title, image, cmap=None, figsize=(8, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

## Helper Function — `ring_score`

After HCT detects circle candidates, each one is scored to determine whether it is actually a stamp or a false detection.

An invisible ring is drawn between 0.75r and 1.15r around the detected circle centre and the fraction of white pixels (stamp ink) within that ring is calculated from the binary mask.

- **High score** — the ring falls on white stamp ink pixels, meaning the detected circle aligns with an actual stamp
- **Low score** — the ring falls mostly on black pixels, meaning the detected circle is a false detection around a non-stamp region such as a border or logo

**Returns:** a float between 0 and 1 — where 1 means the entire ring is stamp ink and 0 means none of it is

In [ ]:
def ring_score(mask, x, y, r):
    h, w = mask.shape[:2]

    yy, xx = np.ogrid[:h, :w]

    dist = np.sqrt((xx - x) ** 2 + (yy - y) ** 2)

    # circular ring region
    ring = (dist >= r * 0.75) & (dist <= r * 1.15)

    if np.sum(ring) == 0:
        return 0

    ring_pixels = mask[ring]

    score = np.sum(ring_pixels == 255) / len(ring_pixels)

    return score

## Function — `detect_colored_components`

Takes a raw document image and runs the entire preprocessing pipeline in one go — resizing, HSV thresholding, morphological operations, and connected component analysis.

At the end, regions that are clearly not stamps are filtered out based on three conditions:
- **Area < 40 px²** — too small, likely noise
- **Compactness < 0.45** — too elongated, likely a text stroke
- **Density < 0.08** — too sparse, likely scattered pixels that accidentally passed the HSV threshold

The remaining regions are the most likely stamp candidates, sorted by area largest first since the stamp is usually the largest coloured region in the document.

**Returns:** `(image_resized, mask_opened, labels, components)` where `components` is a list of dicts each containing `x, y, w, h, area, density, compactness, aspect_ratio`

In [ ]:
def detect_colored_components(image_bgr, display_width=900):
    original_h, original_w = image_bgr.shape[:2]

    scale = display_width / original_w
    resized_h = int(original_h * scale)

    image_resized = cv2.resize(image_bgr, (display_width, resized_h))
    image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]

    hue_mask = cv2.inRange(h, 95, 165)
    sat_mask = cv2.inRange(s, 35, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)

    # MUCH smaller closing kernel
    kernel_close = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3)
    )

    mask_closed = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel_close
    )

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask_opened = cv2.morphologyEx(mask_closed, cv2.MORPH_OPEN, kernel_open)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask_opened,
        connectivity=8
    )

    components = []

    for i in range(1, num_labels):

        x, y, w, h, area = stats[i]

        if area < 40:
            continue

        aspect_ratio = w / h if h > 0 else 0

        # Bounding box area
        bbox_area = w * h

        # Component density
        density = area / bbox_area if bbox_area > 0 else 0

        # Prefer compact/circular regions
        compactness_score = min(w, h) / max(w, h)

        # Reject long text-like regions
        if compactness_score < 0.45:
            continue

        # Reject sparse regions
        if density < 0.08:
            continue

        components.append({
            "label": i,
            "x": int(x),
            "y": int(y),
            "w": int(w),
            "h": int(h),
            "area": int(area),
            "density": float(density),
            "compactness": float(compactness_score),
            "aspect_ratio": float(aspect_ratio)
        })

    components = sorted(
        components,
        key=lambda c: c["area"],
        reverse=True
    )

    return image_resized, mask_opened, labels, components

## Function — `visualize_components`

Draws a green bounding box around each detected component on the image and labels it with its rank number in blue (0 = largest area, 1 = second largest, and so on). The position, size, and shape properties of each component are also printed below the image.

Used to visually verify that the detected components correspond to the actual stamp region before passing them to the next stage.

In [ ]:
def visualize_components(image_resized, components, top_k=10):
    output = image_resized.copy()

    for idx, c in enumerate(components[:top_k]):
        x, y, w, h = c["x"], c["y"], c["w"], c["h"]

        cv2.rectangle(
            output,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            output,
            str(idx),
            (x, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 14))
    plt.imshow(output_rgb)
    plt.title("Colored Components")
    plt.axis("off")
    plt.show()

    for idx, c in enumerate(components[:top_k]):
        print(
            f"{idx}: x={c['x']}, y={c['y']}, w={c['w']}, h={c['h']}, "
            f"area={c['area']}, aspect={c['aspect_ratio']:.2f}"
        )

## Test on Single Image — Component Detection

Loads a single forged sample image and runs `detect_colored_components` to visually verify that the HSV mask and component detection are working correctly before proceeding to HCT.

> ⚠️ Update `sample_path` to a valid image from your dataset.

In [ ]:
sample_path = Path(
    r"H:\UOR\7th sem\Computer Vision\final_dataset\class_1_forged\300dpi-048.png"
)

image_bgr = cv2.imread(str(sample_path))

image_resized, mask_opened, labels, components = detect_colored_components(image_bgr)

print("Number of colored components:", len(components))

show_image("HSV + Morphology Mask", mask_opened, cmap="gray")

visualize_components(
    image_resized,
    components,
    top_k=10
)

## Function — `crop_component_region`

Crops a small rectangular region around a detected component from the full resized image, with extra padding added on each side (`pad_x=120` horizontally, `pad_y=60` vertically) to ensure the full stamp circle is included even if the bounding box slightly underestimates it.

This crop is passed to HCT instead of the full document image — since running HCT on the full image would detect many false circles from other circular elements such as logos, borders, and table corners. Limiting HCT to a small region around the detected component significantly reduces false detections.

In [ ]:
def crop_component_region(image_resized, component, pad_x=120, pad_y=60):
    img_h, img_w = image_resized.shape[:2]

    x = component["x"]
    y = component["y"]
    w = component["w"]
    h = component["h"]

    x1 = max(x - pad_x, 0)
    y1 = max(y - pad_y, 0)
    x2 = min(x + w + pad_x, img_w)
    y2 = min(y + h + pad_y, img_h)

    crop = image_resized[y1:y2, x1:x2]

    return crop, (x1, y1, x2, y2)

## Visualise Component Crop

Takes the top ranked component (`components[0]`) and crops the region around it with 160px horizontal and 80px vertical padding. The crop is displayed to visually verify that the full stamp circle is captured within the region before passing it to HCT.

In [ ]:
selected_component = components[0]

component_crop, bbox = crop_component_region(
    image_resized,
    selected_component,
    pad_x=160,
    pad_y=80
)

component_crop_rgb = cv2.cvtColor(component_crop, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.imshow(component_crop_rgb)
plt.title("Component 0 Crop with Larger Padding")
plt.axis("off")
plt.show()

print("Bounding box:", bbox)
print("Crop shape:", component_crop.shape)

## HSV Mask and Canny Edges on Crop

The HSV thresholding and morphological closing are applied again but this time only on the cropped region. Working on a smaller region produces cleaner, more precise edges for HCT compared to using the mask from the full image.

A Gaussian blur is then applied to smooth out small variations in the mask caused by uneven ink or scan artefacts — so that Canny only detects the significant edges like the outer ring of the stamp and not every tiny imperfection inside it.

Two images are displayed — the binary mask and the edge map — to visually verify the quality before HCT is applied.

| Canny parameter | Value |
|---|---|
| Low threshold | 40 |
| High threshold | 120 |

In [ ]:
# Convert crop to HSV
crop_hsv = cv2.cvtColor(component_crop, cv2.COLOR_BGR2HSV)

h = crop_hsv[:, :, 0]
s = crop_hsv[:, :, 1]

# HSV threshold
hue_mask = cv2.inRange(h, 95, 165)
sat_mask = cv2.inRange(s, 35, 255)

crop_mask = cv2.bitwise_and(hue_mask, sat_mask)

# Morphology
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5, 5)
)

crop_mask = cv2.morphologyEx(
    crop_mask,
    cv2.MORPH_CLOSE,
    kernel
)

crop_mask = cv2.GaussianBlur(
    crop_mask,
    (5, 5),
    1.5
)

# Canny
crop_edges = cv2.Canny(
    crop_mask,
    40,
    120
)

show_image(
    "Crop Mask",
    crop_mask,
    cmap="gray"
)

show_image(
    "Crop Edges",
    crop_edges,
    cmap="gray"
)

## Apply Hough Circle Transform

Applies HCT on the Canny edge map of the crop to detect circles. For each detected circle a green outline is drawn with a blue centre point, labelled by its index number.

| Parameter | Value | Purpose |
|---|---|---|
| `dp` | 1.2 | Accumulator resolution slightly lower than image resolution — speeds up detection |
| `minDist` | 40 px | Minimum distance between two circle centres — prevents duplicate detections of the same stamp |
| `param1` | 120 | Upper Canny threshold used internally by HCT |
| `param2` | 18 | Sensitivity threshold — lower value detects more circles but risks more false positives |
| `minRadius` | 15 px | Minimum circle size to search for within the crop |
| `maxRadius` | 80 px | Maximum circle size to search for within the crop |

In [ ]:
crop_circles = cv2.HoughCircles(
    crop_edges,
    cv2.HOUGH_GRADIENT,
    dp=1.2,
    minDist=40,
    param1=120,
    param2=18,
    minRadius=15,
    maxRadius=80
)

crop_output = component_crop.copy()

if crop_circles is not None:

    crop_circles = np.round(
        crop_circles[0, :]
    ).astype("int")

    print("Detected circles:", len(crop_circles))

    for idx, (x, y, r) in enumerate(crop_circles):

        cv2.circle(
            crop_output,
            (x, y),
            r,
            (0, 255, 0),
            2
        )

        cv2.circle(
            crop_output,
            (x, y),
            2,
            (255, 0, 0),
            3
        )

        cv2.putText(
            crop_output,
            str(idx),
            (x + 5, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

else:
    print("No circles detected")

crop_output_rgb = cv2.cvtColor(
    crop_output,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(8, 8))
plt.imshow(crop_output_rgb)
plt.title("Hough Circles Inside Component Crop")
plt.axis("off")
plt.show()

## Function — `crop_top_components`

Crops a region around each of the top k components (default k=3) from the ranked component list using `crop_component_region`. All three crops are passed to HCT rather than just the top one — since the actual stamp is not always the largest coloured region and may be ranked 2nd or 3rd by area.

**Returns:** a list of dicts each containing the crop image, bounding box coordinates, component index, and original component properties

In [ ]:
def crop_top_components(image_resized, components, top_k=3, pad_x=120, pad_y=80):
    crops = []

    for idx, component in enumerate(components[:top_k]):
        crop, bbox = crop_component_region(
            image_resized,
            component,
            pad_x=pad_x,
            pad_y=pad_y
        )

        crops.append({
            "component_index": idx,
            "crop": crop,
            "bbox": bbox,
            "component": component
        })

    return crops

## 12. Visualise Top Candidate Crops

Displays the top 3 component crops for visual inspection before running the full pipeline.

In [ ]:
candidate_crops = crop_top_components(
    image_resized,
    components,
    top_k=3,
    pad_x=120,
    pad_y=80
)

for item in candidate_crops:
    crop_rgb = cv2.cvtColor(item["crop"], cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6, 6))
    plt.imshow(crop_rgb)
    plt.title(f"Candidate Crop {item['component_index']}")
    plt.axis("off")
    plt.show()

## Function — `detect_stamp_candidates`

The complete end-to-end pipeline function that combines all previous steps into a single call. Given a raw document image it runs the following steps internally:

1. Resizes the image to `display_width` (default 900px)
2. Applies HSV thresholding on Hue (95–165) and Saturation (35–255)
3. Applies morphological closing (7×7) then opening (3×3) to clean the mask
4. Applies Gaussian blur then Canny edge detection on the cleaned mask
5. Runs HCT directly on the full edge map without cropping individual components first
6. Scores each detected circle using `ring_score` and discards any scoring below 0.08
7. Returns remaining candidates sorted by ring score highest first

**Returns:** `(image_resized, mask_opened, edges, candidates)`

In [ ]:
def detect_stamp_candidates(image_bgr, display_width=900):
    original_h, original_w = image_bgr.shape[:2]

    scale = display_width / original_w
    resized_h = int(original_h * scale)

    image_resized = cv2.resize(
        image_bgr,
        (display_width, resized_h)
    )

    image_hsv = cv2.cvtColor(
        image_resized,
        cv2.COLOR_BGR2HSV
    )

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]

    hue_mask = cv2.inRange(h, 95, 165)
    sat_mask = cv2.inRange(s, 35, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)

    kernel_close = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (7, 7)
    )

    mask_closed = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel_close
    )

    kernel_open = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3)
    )

    mask_opened = cv2.morphologyEx(
        mask_closed,
        cv2.MORPH_OPEN,
        kernel_open
    )

    blurred = cv2.GaussianBlur(
        mask_opened,
        (5, 5),
        1.5
    )

    edges = cv2.Canny(
        blurred,
        40,
        120
    )

    circles = cv2.HoughCircles(
        edges,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=50,
        param1=120,
        param2=22,
        minRadius=15,
        maxRadius=70
    )

    if circles is None:
        return image_resized, mask_opened, edges, []

    circles = np.round(circles[0, :]).astype("int")

    candidates = []

    for x, y, r in circles:

        rs = ring_score(
            mask_opened,
            x,
            y,
            r
        )

        if rs < 0.08:
            continue

        candidates.append({
            "x": int(x),
            "y": int(y),
            "r": int(r),
            "ring_score": float(rs)
        })

    candidates = sorted(
        candidates,
        key=lambda c: c["ring_score"],
        reverse=True
    )

    return image_resized, mask_opened, edges, candidates

## 14. Function: `visualize_candidates`

Draws detected circles (green) and centre points (red) on the image, labelled by rank index.

In [ ]:
def visualize_candidates(image_resized, candidates, top_k=10):

    output = image_resized.copy()

    for idx, c in enumerate(candidates[:top_k]):

        x = c["x"]
        y = c["y"]
        r = c["r"]

        cv2.circle(
            output,
            (x, y),
            r,
            (0, 255, 0),
            2
        )

        cv2.circle(
            output,
            (x, y),
            2,
            (255, 0, 0),
            3
        )

        cv2.putText(
            output,
            str(idx),
            (x + 5, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    output_rgb = cv2.cvtColor(
        output,
        cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(10, 14))

    plt.imshow(output_rgb)

    plt.title("Detected Stamp Candidates")

    plt.axis("off")

    plt.show()

    for idx, c in enumerate(candidates[:top_k]):

        print(
            f"{idx}: "
            f"x={c['x']}, "
            f"y={c['y']}, "
            f"r={c['r']}, "
            f"ring_score={c['ring_score']:.4f}"
        )

## 15. Test on Single Image — Full Pipeline

Runs `detect_stamp_candidates` on the sample image and displays the mask, edges, and circle candidates.


In [ ]:
sample_path = Path(
    r"H:\UOR\7th sem\Computer Vision\final_dataset\class_1_forged\300dpi-048.png"
)

image_bgr = cv2.imread(str(sample_path))

image_resized, mask_opened, edges, candidates = detect_stamp_candidates(image_bgr)

print("Number of candidates:", len(candidates))

show_image(
    "Mask Opened",
    mask_opened,
    cmap="gray"
)

show_image(
    "Canny Edges",
    edges,
    cmap="gray"
)

visualize_candidates(
    image_resized,
    candidates,
    top_k=10
)

## 16. Batch Test — 5 Genuine + 5 Forged

Runs the full pipeline on 10 images (5 genuine, 5 forged) and displays candidates for each. This stress-tests parameter robustness across different stamp impressions, scan resolutions, and forgery types.


In [ ]:
DATASET_ROOT = Path(r"H:\UOR\7th sem\Computer Vision\final_dataset")

# Pick 5 genuine and 5 forged images
genuine_images = list((DATASET_ROOT / "class_0_genuine").rglob("*.png"))[:5]
forged_images = list((DATASET_ROOT / "class_1_forged").rglob("*.png"))[:5]

test_images = genuine_images + forged_images

print("Total test images:", len(test_images))

In [ ]:
for img_path in test_images:
    print("=" * 100)
    print("Testing:", img_path)

    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        print("Could not read image")
        continue

    image_resized, mask_opened, edges, candidates = detect_stamp_candidates(image_bgr)

    print("Number of candidates:", len(candidates))

    if len(candidates) == 0:
        print("No candidates found")
        continue

    visualize_candidates(
        image_resized,
        candidates,
        top_k=10
    )